## Código para Download do dataset

In [ ]:
import os
import time
import random
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

from SciServer import SkyServer


# ============================================================
# CONFIGURAÇÃO
# ============================================================

CLASSES = ["GALAXY", "QSO", "STAR"]

# Quantidade máxima por classe
MAX_PER_CLASS = 10_000

# Divisão do dataset
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Seed para garantir que o split seja sempre reproduzível
SEED = 42

# Configuração das imagens SDSS
IMAGE_WIDTH = 224
IMAGE_HEIGHT = 224

# Aproximadamente a escala de pixel do SDSS Legacy Imaging
IMAGE_SCALE = 0.396

# Data Release utilizada
DATA_RELEASE = "DR20"

# Intervalo entre requisições
REQUEST_DELAY = 0.05


# ============================================================
# ARGUMENTOS
# ============================================================

def parse_args():

    parser = argparse.ArgumentParser(
        description="Baixa imagens SDSS usando um CSV de objetos."
    )

    parser.add_argument(
        "--csv",
        required=True,
        help="Caminho para o CSV contendo os objetos."
    )

    parser.add_argument(
        "--output",
        default="dataset",
        help="Diretório onde o dataset será salvo."
    )

    parser.add_argument(
        "--max-per-class",
        type=int,
        default=MAX_PER_CLASS,
        help="Quantidade máxima de objetos por classe."
    )

    parser.add_argument(
        "--width",
        type=int,
        default=IMAGE_WIDTH,
        help="Largura da imagem."
    )

    parser.add_argument(
        "--height",
        type=int,
        default=IMAGE_HEIGHT,
        help="Altura da imagem."
    )

    parser.add_argument(
        "--scale",
        type=float,
        default=IMAGE_SCALE,
        help="Escala em arcsec/pixel."
    )

    parser.add_argument(
        "--force",
        action="store_true",
        help="Baixa novamente imagens que já existem."
    )

    return parser.parse_args()


# ============================================================
# LEITURA DO CSV
# ============================================================

def load_csv(csv_path):

    print(f"\nLendo CSV: {csv_path}")

    df = pd.read_csv(csv_path)

    print(f"Objetos encontrados: {len(df)}")

    print("\nColunas:")
    for column in df.columns:
        print(f"  - {column}")

    return df


# ============================================================
# NORMALIZAÇÃO DAS COLUNAS
# ============================================================

def prepare_dataframe(df):

    # Remove espaços dos nomes
    df.columns = [str(c).strip() for c in df.columns]

    required_columns = [
        "ra",
        "dec",
        "spectralClass"
    ]

    missing = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Colunas obrigatórias não encontradas: {missing}"
        )

    # Normaliza classes
    df["spectralClass"] = (
        df["spectralClass"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # Mantém somente as classes desejadas
    df = df[
        df["spectralClass"].isin(CLASSES)
    ].copy()

    # Remove RA/DEC inválidos
    df["ra"] = pd.to_numeric(
        df["ra"],
        errors="coerce"
    )

    df["dec"] = pd.to_numeric(
        df["dec"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["ra", "dec"]
    )

    # Remove coordenadas inválidas
    df = df[
        (df["ra"] >= 0) &
        (df["ra"] <= 360) &
        (df["dec"] >= -90) &
        (df["dec"] <= 90)
    ]

    # Remove duplicatas
    if "objID" in df.columns:
        df = df.drop_duplicates(
            subset=["objID"]
        )
    else:
        df = df.drop_duplicates(
            subset=["ra", "dec"]
        )

    print("\nObjetos por classe:")

    print(
        df["spectralClass"]
        .value_counts()
        .sort_index()
    )

    return df


# ============================================================
# SPLIT DO DATASET
# ============================================================

def create_splits(
    df,
    max_per_class
):

    rng = np.random.default_rng(SEED)

    datasets = []

    for class_name in CLASSES:

        class_df = df[
            df["spectralClass"] == class_name
        ].copy()

        if len(class_df) < max_per_class:

            print(
                f"\nAviso: {class_name} possui "
                f"{len(class_df)} objetos."
            )

            print(
                f"Serão utilizados todos os objetos disponíveis."
            )

        else:

            # Amostragem determinística
            indices = rng.choice(
                len(class_df),
                size=max_per_class,
                replace=False
            )

            class_df = class_df.iloc[
                indices
            ].copy()

        # Embaralhamento
        class_df = class_df.sample(
            frac=1,
            random_state=SEED
        ).reset_index(drop=True)

        n = len(class_df)

        train_end = int(
            n * TRAIN_RATIO
        )

        val_end = train_end + int(
            n * VAL_RATIO
        )

        class_df.loc[
            :train_end - 1,
            "split"
        ] = "train"

        class_df.loc[
            train_end:val_end - 1,
            "split"
        ] = "val"

        class_df.loc[
            val_end:,
            "split"
        ] = "test"

        datasets.append(class_df)

    result = pd.concat(
        datasets,
        ignore_index=True
    )

    return result


# ============================================================
# CRIA DIRETÓRIOS
# ============================================================

def create_directories(output):

    output = Path(output)

    for split in [
        "train",
        "val",
        "test"
    ]:

        for class_name in CLASSES:

            path = (
                output /
                split /
                class_name
            )

            path.mkdir(
                parents=True,
                exist_ok=True
            )

    return output


# ============================================================
# NOME DO ARQUIVO
# ============================================================

def get_object_id(row, index):

    if "objID" in row.index:

        value = row["objID"]

        if pd.notna(value):

            return str(
                int(float(value))
            )

    return f"object_{index:08d}"


# ============================================================
# DOWNLOAD DA IMAGEM
# ============================================================

def download_image(
    ra,
    dec,
    output_path,
    width,
    height,
    scale,
    force=False
):

    output_path = Path(output_path)

    # Evita baixar novamente
    if (
        output_path.exists()
        and not force
    ):

        return True, "exists"

    try:

        image = SkyServer.getJpegImgCutout(
            ra=float(ra),
            dec=float(dec),
            scale=float(scale),
            width=int(width),
            height=int(height),
            dataRelease=DATA_RELEASE
        )

        if image is None:

            return False, "empty"

        image = np.asarray(image)

        if image.size == 0:

            return False, "empty"

        # Garante formato RGB
        if image.ndim == 2:

            image = np.stack(
                [image] * 3,
                axis=-1
            )

        elif (
            image.ndim == 3
            and image.shape[2] > 3
        ):

            image = image[:, :, :3]

        # Converte para uint8
        if image.dtype != np.uint8:

            image = np.clip(
                image,
                0,
                255
            ).astype(np.uint8)

        pil_image = Image.fromarray(
            image
        ).convert("RGB")

        pil_image.save(
            output_path,
            format="JPEG",
            quality=95
        )

        return True, "downloaded"

    except Exception as error:

        return False, str(error)


# ============================================================
# DOWNLOAD DO DATASET
# ============================================================

def download_dataset(
    df,
    output,
    width,
    height,
    scale,
    force
):

    output = Path(output)

    total = len(df)

    success = 0
    existing = 0
    failed = 0

    failures = []

    print(
        f"\nIniciando download de {total} imagens..."
    )

    for index, row in df.iterrows():

        class_name = row[
            "spectralClass"
        ]

        split = row[
            "split"
        ]

        object_id = get_object_id(
            row,
            index
        )

        filename = (
            f"{object_id}.jpg"
        )

        output_path = (
            output /
            split /
            class_name /
            filename
        )

        ok, status = download_image(
            ra=row["ra"],
            dec=row["dec"],
            output_path=output_path,
            width=width,
            height=height,
            scale=scale,
            force=force
        )

        if ok:

            if status == "exists":

                existing += 1

            else:

                success += 1

        else:

            failed += 1

            failures.append({

                "index": index,

                "objID": row.get(
                    "objID",
                    ""
                ),

                "ra": row["ra"],

                "dec": row["dec"],

                "spectralClass":
                    class_name,

                "split":
                    split,

                "error":
                    status
            })

        processed = (
            success +
            existing +
            failed
        )

        if (
            processed % 100 == 0
            or processed == total
        ):

            print(
                f"[{processed}/{total}] "
                f"baixadas={success} "
                f"existentes={existing} "
                f"falhas={failed}"
            )

        time.sleep(
            REQUEST_DELAY
        )

    # Salva log de falhas
    if failures:

        failures_df = pd.DataFrame(
            failures
        )

        failures_df.to_csv(
            output /
            "download_failures.csv",
            index=False
        )

    print("\n==============================")
    print("DOWNLOAD FINALIZADO")
    print("==============================")

    print(
        f"Novas imagens: {success}"
    )

    print(
        f"Já existentes: {existing}"
    )

    print(
        f"Falhas: {failed}"
    )

    return failures


# ============================================================
# SALVA MANIFESTO
# ============================================================

def save_manifest(
    df,
    output
):

    manifest_path = (
        Path(output) /
        "dataset_manifest.csv"
    )

    df.to_csv(
        manifest_path,
        index=False
    )

    print(
        f"\nManifesto salvo em:"
        f"\n{manifest_path}"
    )


# ============================================================
# ESTATÍSTICAS
# ============================================================

def print_statistics(df):

    print("\n==============================")
    print("ESTATÍSTICAS DO DATASET")
    print("==============================")

    print("\nPor classe:")

    print(
        df[
            "spectralClass"
        ].value_counts()
        .sort_index()
    )

    print("\nPor split:")

    print(
        df[
            "split"
        ].value_counts()
        .sort_index()
    )

    print(
        "\nClasse × Split:"
    )

    print(
        pd.crosstab(
            df["spectralClass"],
            df["split"]
        )
    )


# ============================================================
# MAIN
# ============================================================

def main():

    args = parse_args()

    print(
        "\n======================================"
    )

    print(
        " SDSS DR20 IMAGE DATASET BUILDER"
    )

    print(
        "======================================"
    )

    # --------------------------------------------------------
    # CSV
    # --------------------------------------------------------

    df = load_csv(
        args.csv
    )

    # --------------------------------------------------------
    # PREPARAÇÃO
    # --------------------------------------------------------

    df = prepare_dataframe(
        df
    )

    # --------------------------------------------------------
    # SPLIT
    # --------------------------------------------------------

    df = create_splits(
        df,
        args.max_per_class
    )

    print_statistics(
        df
    )

    # --------------------------------------------------------
    # DIRETÓRIOS
    # --------------------------------------------------------

    output = create_directories(
        args.output
    )

    # --------------------------------------------------------
    # MANIFEST
    # --------------------------------------------------------

    save_manifest(
        df,
        output
    )

    # --------------------------------------------------------
    # DOWNLOAD
    # --------------------------------------------------------

    download_dataset(
        df=df,
        output=output,
        width=args.width,
        height=args.height,
        scale=args.scale,
        force=args.force
    )

    print(
        "\nPipeline concluído."
    )


if __name__ == "__main__":

    main()